In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

SILVER_CORE_TABLE = f"{CATALOG}.{SCHEMA}.silver_events_core"
ACTOR_IDENTITY_TABLE = f"{CATALOG}.{SCHEMA}.dim_actor_identity"
GOLD_REPO_DAILY_TABLE = f"{CATALOG}.{SCHEMA}.gold_repo_daily_activity"

In [0]:
actor_activity_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(
        F.col("actor_id").isNotNull() &
        F.col("actor_login").isNotNull()
    )
    .groupBy(
        "actor_id",
        "actor_login"
    )
    .agg(
        F.count("*").alias("event_count")
    )
)

print(
    "Distinct actor identities:",
    actor_activity_df.count()
)

In [0]:
bot_candidate_df = (
    actor_activity_df
    .withColumn(
        "login_lower",
        F.lower("actor_login")
    )
    .withColumn(
        "bot_pattern",
        F.when(
            F.col("login_lower").endswith("[bot]"),
            "github_bot_suffix"
        )
        .when(
            F.col("login_lower").endswith("-bot"),
            "dash_bot_suffix"
        )
        .when(
            F.col("login_lower").endswith("_bot"),
            "underscore_bot_suffix"
        )
        .when(
            F.col("login_lower").startswith("bot-"),
            "bot_prefix"
        )
    )
    .filter(
        F.col("bot_pattern").isNotNull()
    )
)

In [0]:
bot_candidate_df.groupBy(
    "bot_pattern"
).agg(
    F.countDistinct("actor_id").alias("distinct_actors"),
    F.sum("event_count").alias("events")
).orderBy(
    F.desc("events")
).show(truncate=False)

In [0]:
bot_candidate_df.select(
    "actor_id",
    "actor_login",
    "bot_pattern",
    "event_count"
).orderBy(
    F.desc("event_count")
).show(
    50,
    truncate=False
)

In [0]:
actor_identity_df = (
    actor_activity_df
    .withColumn(
        "actor_type",
        F.when(
            F.lower("actor_login").endswith("[bot]"),
            F.lit("bot")
        ).otherwise(
            F.lit("human_or_unknown")
        )
    )
)

In [0]:
actor_identity_df.groupBy(
    "actor_type"
).agg(
    F.countDistinct("actor_id").alias("distinct_actors"),
    F.sum("event_count").alias("events")
).show(truncate=False)

In [0]:
bot_activity_summary_df = (
    actor_identity_df
    .groupBy()
    .agg(
        F.sum("event_count").alias("total_events"),

        F.sum(
            F.when(
                F.col("actor_type") == "bot",
                F.col("event_count")
            ).otherwise(0)
        ).alias("bot_events"),

        F.countDistinct("actor_id")
            .alias("total_actors"),

        F.countDistinct(
            F.when(
                F.col("actor_type") == "bot",
                F.col("actor_id")
            )
        ).alias("bot_actors")
    )
    .withColumn(
        "bot_event_share",
        F.col("bot_events") / F.col("total_events")
    )
    .withColumn(
        "bot_actor_share",
        F.col("bot_actors") / F.col("total_actors")
    )
)

bot_activity_summary_df.show(truncate=False)

In [0]:
actor_login_variation_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(F.col("actor_id").isNotNull())
    .groupBy("actor_id")
    .agg(
        F.countDistinct("actor_login")
            .alias("distinct_logins")
    )
)

print(
    "Actor IDs with multiple observed logins:",
    actor_login_variation_df
    .filter(F.col("distinct_logins") > 1)
    .count()
)


In [0]:
actor_login_changes_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(
        F.col("actor_id").isNotNull() &
        F.col("actor_login").isNotNull()
    )
    .groupBy("actor_id")
    .agg(
        F.collect_set("actor_login").alias("observed_logins"),
        F.countDistinct("actor_login").alias("distinct_logins"),
        F.min("event_ts").alias("first_seen"),
        F.max("event_ts").alias("last_seen")
    )
    .filter(
        F.col("distinct_logins") > 1
    )
)

actor_login_changes_df.show(
    truncate=False
)

In [0]:
silver_core_identity_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(F.col("actor_id").isNotNull())
)

actor_identity_dim_df = (
    silver_core_identity_df
    .groupBy("actor_id")
    .agg(
        F.max_by("actor_login", "event_ts")
            .alias("actor_login"),

        F.min("event_ts")
            .alias("first_seen"),

        F.max("event_ts")
            .alias("last_seen"),

        F.countDistinct("actor_login")
            .alias("observed_login_count"),

        F.max(
            F.when(
                F.lower("actor_login").endswith("[bot]"),
                F.lit(1)
            ).otherwise(F.lit(0))
        ).alias("bot_flag")
    )
    .withColumn(
        "actor_type",
        F.when(
            F.col("bot_flag") == 1,
            F.lit("bot")
        ).otherwise(
            F.lit("human_or_unknown")
        )
    )
    .drop("bot_flag")
)

In [0]:
print(
    "Actor identity rows:",
    actor_identity_dim_df.count()
)

duplicate_actor_ids = (
    actor_identity_dim_df
    .groupBy("actor_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate actor IDs:",
    duplicate_actor_ids
)

actor_identity_dim_df.groupBy(
    "actor_type"
).count().show()

In [0]:
(
    actor_identity_dim_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ACTOR_IDENTITY_TABLE)
)

In [0]:
persisted_identity_df = spark.table(ACTOR_IDENTITY_TABLE)

print(
    "Persisted actor identities:",
    persisted_identity_df.count()
)

print(
    "Persisted duplicate actor IDs:",
    persisted_identity_df
    .groupBy("actor_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [0]:
bot_aware_activity_df = (
    spark.table(SILVER_CORE_TABLE)
    .filter(
        F.col("repo_id").isNotNull() &
        F.col("actor_id").isNotNull()
    )
    .withColumn(
        "event_date",
        F.to_date("event_ts")
    )
    .join(
        actor_identity_dim_df.select(
            "actor_id",
            "actor_type"
        ),
        on="actor_id",
        how="left"
    )
    .groupBy(
        "repo_id",
        "event_date"
    )
    .agg(
        F.sum(
            (F.col("actor_type") == "bot").cast("int")
        ).alias("bot_events"),

        F.sum(
            (F.col("actor_type") == "human_or_unknown").cast("int")
        ).alias("human_or_unknown_events"),

        F.countDistinct(
            F.when(
                F.col("actor_type") == "bot",
                F.col("actor_id")
            )
        ).alias("bot_contributors"),

        F.countDistinct(
            F.when(
                F.col("actor_type") == "human_or_unknown",
                F.col("actor_id")
            )
        ).alias("human_or_unknown_contributors")
    )
)

In [0]:
bot_aware_activity_df.select(
    F.sum("bot_events").alias("bot_events"),
    F.sum("human_or_unknown_events").alias("human_or_unknown_events")
).show()

print(
    "Bot-aware repo/day rows:",
    bot_aware_activity_df.count()
)

In [0]:
gold_repo_daily_df = (
    spark.table(GOLD_REPO_DAILY_TABLE)
    .join(
        bot_aware_activity_df,
        on=["repo_id", "event_date"],
        how="left"
    )
    .fillna(
        0,
        subset=[
            "bot_events",
            "human_or_unknown_events",
            "bot_contributors",
            "human_or_unknown_contributors"
        ]
    )
)

In [0]:
print(
    "Bot-enriched Gold rows:",
    gold_repo_daily_df.count()
)

duplicate_gold_keys = (
    gold_repo_daily_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate repo/date keys:",
    duplicate_gold_keys
)

gold_repo_daily_df.select(
    F.sum("bot_events").alias("bot_events"),
    F.sum("human_or_unknown_events").alias("human_or_unknown_events"),
    F.sum("bot_contributors").alias("bot_contributors_repo_day_sum"),
    F.sum("human_or_unknown_contributors").alias("human_contributors_repo_day_sum")
).show()

In [0]:
gold_reconciliation_df = (
    gold_repo_daily_df
    .select(
        F.sum("total_events").alias("gold_total_events"),
        F.sum("bot_events").alias("bot_events"),
        F.sum("human_or_unknown_events").alias("human_or_unknown_events")
    )
    .withColumn(
        "classified_event_total",
        F.col("bot_events") +
        F.col("human_or_unknown_events")
    )
    .withColumn(
        "reconciled",
        F.col("gold_total_events") ==
        F.col("classified_event_total")
    )
)

gold_reconciliation_df.show(truncate=False)

In [0]:
(
    gold_repo_daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REPO_DAILY_TABLE)
)

In [0]:
final_gold_df = spark.table(GOLD_REPO_DAILY_TABLE)

print(
    "Final Gold rows:",
    final_gold_df.count()
)

print(
    "Final duplicate repo/date keys:",
    final_gold_df
    .groupBy("repo_id", "event_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

final_gold_df.select(
    F.sum("total_events").alias("total_events"),
    F.sum("bot_events").alias("bot_events"),
    F.sum("human_or_unknown_events").alias("human_or_unknown_events")
).show()

## Phase 11 conclusions

Actor identity is modeled using `actor_id` as the stable identity key. Login names are treated as mutable attributes.

A conservative bot classification rule was applied:

- Login ending with `[bot]` -> `bot`
- All other actors -> `human_or_unknown`

Results:

- 55,396 unique actor identities
- 698 classified bots
- 54,698 human-or-unknown actors
- 23,139 bot-generated events
- 593,547 human-or-unknown events

Two actor IDs were observed with multiple login names, confirming that login should not be used as the identity key.

Bot-aware metrics were added to the repository-day Gold table:

- bot_events
- human_or_unknown_events
- bot_contributors
- human_or_unknown_contributors

Repository-level Gold activity reconciles exactly with classified activity:

- Gold repository events: 616,686
- Classified repository events: 616,686
- Reconciliation: true

One Silver event with a null repository ID remains intentionally excluded from repository-level Gold metrics.